In [180]:
import os
import numpy as np
import sympy as sp
import torch

from SymbolicMathematics.src.envs import set_cover_utils
from src.utils import AttrDict
from src.envs import build_env
from src.model import build_modules

from src.utils import to_cuda
from src.envs.sympy_utils import simplify
from src.envs.set_cover_utils import validate_set_cover_solution

from src.envs import char_sp
from src.envs.char_sp import CharSPEnvironment
from itertools import combinations

import json
import matplotlib.pyplot as plt

## Required Functions

In [181]:
def is_covering(matrix, selected_columns, rows):
    covered = set()
    for col in selected_columns:
        covered.update(np.where(matrix[:, col])[0])
    return len(covered) == rows

In [182]:
def set_cover_bruteforce(matrix):
    rows, cols = matrix.shape
    for num_columns in range(1, cols + 1):
        for selected_columns in combinations(range(cols), num_columns):
            if is_covering(matrix, selected_columns, rows):
                return selected_columns
    return None

In [183]:
def list_to_tokens(lst):
    return [str(c) for c in str(lst) if str(c) != ' ']

In [184]:
def convert_token_sequence_to_list_of_lists(token_sequence):
    list_of_lists = []  # This will store the final list of lists
    current_list = []  # This will store the current list being constructed
    depth = 0  # Variable to keep track of the depth of nesting
    inner_list = []  # Initialize inner_list

    if len(token_sequence) == 0:
        return []

    for token in token_sequence:  # Iterate over each token in the token sequence
        if token == '[':  # If an opening bracket is encountered
            if depth > 0:
                inner_list = []  # Start a new inner list
            depth += 1  # Increment the depth of nesting
        elif token == ']':  # If a closing bracket is encountered
            depth -= 1  # Decrement the depth of nesting
            if depth == 0:
                if inner_list:  # Check if the inner list is not empty
                    current_list.append(inner_list)  # Append the inner list to the current list
                list_of_lists.append(current_list)  # Append the constructed list to the final result
                current_list = []  # Reset the current list
            else:
                current_list.append(inner_list)  # Append the inner list to the current list
                inner_list = []  # Start a new inner list
        elif token.isdigit():  # If the token is an integer
            inner_list.append(int(token))  # Convert token to integer and add to the inner list

    if len(list_of_lists) == 0:
        return []
    return list_of_lists[0]  # Return the final list of lists

In [185]:
seed = 1234
def gen_set_cover(max_universe_size=6):
    rng = np.random.RandomState(seed)
    # Generate a random binary matrix
    num_rows = rng.randint(3, max_universe_size + 1) #elements
    num_cols = rng.randint(2, 2*max_universe_size + 1) #subsets
    binary_matrix = np.random.choice([False, True], size=(num_rows, num_cols))  # Randomly generate binary matrix

    for i in range(num_rows):
        if not binary_matrix[i].any():
            binary_matrix[i, np.random.randint(0, num_cols)] = True
    for j in range(num_cols):
        if not binary_matrix[:, j].any():
            binary_matrix[np.random.randint(0, num_rows), j] = True

    # Remove duplicate rows
    binary_matrix = np.unique(binary_matrix, axis=1)

    # Find a set cover solution using brute force
    solution_indices = set_cover_bruteforce(binary_matrix)

    # Convert binary matrix and solution to sequences
    input_sequence = [list(np.where(column)[0]) for column in binary_matrix.T]
    # Map the solution subsets to their corresponding indices in the input sequence
    label_sequence = [subset for idx, subset in enumerate(input_sequence) if idx in solution_indices]

    input_sequence = list_to_tokens(input_sequence)
    label_sequence = list_to_tokens(label_sequence)

    return input_sequence, label_sequence

In [186]:
    seed = 125
    def gen_set_cover2(max_universe_size=6):
        rng = np.random.RandomState(seed)
        universe_list = list(range(10))  # Default universe list from 0 to 20
        selected_elements = np.random.choice(universe_list, max_universe_size, replace=False)

        # Generate a random binary matrix
        num_rows = rng.randint(3, max_universe_size + 1) #elements
        num_cols = rng.randint(2, 2*max_universe_size + 1) #subsets
        binary_matrix = np.random.choice([False, True], size=(num_rows, num_cols))  # Randomly generate binary matrix

        for i in range(num_rows):
            if not binary_matrix[i].any():
                binary_matrix[i, np.random.randint(0, num_cols)] = True
        for j in range(num_cols):
            if not binary_matrix[:, j].any():
                binary_matrix[np.random.randint(0, num_rows), j] = True

        # Remove duplicate rows
        binary_matrix = np.unique(binary_matrix, axis=1)
        # Find a set cover solution using brute force
        solution_indices = set_cover_bruteforce(binary_matrix)

        # Convert binary matrix columns to sequences of selected elements
        input_sequence = []
        for column in binary_matrix.T:
            subset = [selected_elements[row] for row in np.where(column)[0]]
            input_sequence.append(subset)

        label_sequence = [subset for idx, subset in enumerate(input_sequence) if idx in solution_indices]

        input_sequence = list_to_tokens(input_sequence)
        label_sequence = list_to_tokens(label_sequence)


        return input_sequence, label_sequence

In [187]:
def validate_set_cover_solution(problem_set, solution_set, true_output):
    valid_solution_count = 0
    error_counts = {
        "Wrong Format": 0,
        "Subset not in problem set": 0,
        "Solution not covering problem set": 0,
        "Solution is not optimal": 0
    }
    coverage_percentage = 0

    # Check if the solution set is empty
    if not solution_set:
        error_counts["Wrong Format"] += 1
        return valid_solution_count, error_counts, coverage_percentage


    # Check if any subset in the solution set is empty
    for j, subset in enumerate(solution_set):
        if not subset:
            error_counts["Wrong Format"] += 1
            break

    # Check if all subsets in the solution set exist in the problem set
        if subset not in problem_set:
            error_counts["Subset not in problem set"] += 1
            break

    else:
        # Check if the solution set covers the entire problem set
        covered_tokens = set().union(*solution_set)
        problem_tokens = set(token for sublist in problem_set for token in sublist)
        if covered_tokens != problem_tokens:
            coverage_percentage = round(len(covered_tokens) / len(problem_tokens) * 100, 2)
            error_counts["Solution not covering problem set"] += 1

        else:
            if len(solution_set) > len(true_output):
                error_counts["Solution is not optimal"] += 1
            else:
                valid_solution_count += 1

    return valid_solution_count, error_counts, coverage_percentage

In [188]:
#Test
input1, label1 = gen_set_cover(6)
print("input sequence:", input1)
print("label sequence:", label1)

input sequence: ['[', '[', '2', ',', '3', ']', ',', '[', '0', ',', '3', ']', ',', '[', '0', ',', '1', ',', '3', ']', ',', '[', '0', ',', '1', ',', '2', ']', ']']
label sequence: ['[', '[', '2', ',', '3', ']', ',', '[', '0', ',', '1', ',', '3', ']', ']']


## Build environment / Reload model

In [189]:
# Reload Trained model
model_path1 = './Model1/first_train/2spv1sq7hj/checkpoint.pth'
assert os.path.isfile(model_path1)

#model_path2 = './Model2/first_train/5r8wvsegfc/checkpoint.pth'
#assert os.path.isfile(model_path2)


In [190]:
params = AttrDict({

    # environment parameters
    'env_name': 'char_sp',
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,
    'n_variables': 1,
    'n_coefficients': 0,
    'leaf_probs': '0.75,0,0.25,0',
    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,
    'clean_prefix_expr': True,
    'rewrite_functions': '',
    #'tasks': 'prim_fwd',
    'tasks': 'set_cover',
    'operators': 'add:10,sub:3,mul:10,div:5,sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,ln:4,exp:4,sin:4,cos:4,tan:4,asin:1,acos:1,atan:1,sinh:1,cosh:1,tanh:1,asinh:1,acosh:1,atanh:1',

    # model parameters
    'cpu': True,
    'emb_dim': 128,
    'n_enc_layers': 2, #it was 6
    'n_dec_layers': 2, #it was 6
    'n_heads': 2, #it was 8
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,
    'reload_model': model_path1,

})

In [191]:
env = build_env(params)
#x = env.local_dict['x']

In [192]:
modules = build_modules(env, params)
encoder = modules['encoder']
decoder = modules['decoder']

In [193]:
problem_set_instance = [[1,2],[1,2,3],[2],[4]]
problem_instance_tokens = list_to_tokens(problem_set_instance)
print(problem_instance_tokens)

['[', '[', '1', ',', '2', ']', ',', '[', '1', ',', '2', ',', '3', ']', ',', '[', '2', ']', ',', '[', '4', ']', ']']


### Encode input

In [216]:
# Loop through the specified number of iterations
num_iterations = 1
total_valid_answers_beam1 = 0
total_valid_answers_beam10 = 0

for i in range(num_iterations):
    # Generate the input and label sequences
    input_sequence, label_sequence = gen_set_cover()
    
    # Encode the input sequence
    x1_prefix = input_sequence
    x1 = torch.LongTensor(
        [env.eos_index] +
        [env.word2id[w] for w in x1_prefix] +
        [env.eos_index]
    ).view(-1, 1)
    len1 = torch.LongTensor([len(x1)])
    # x1, len1 = to_cuda(x1, len1)  # Uncomment if you are using GPU
    with torch.no_grad():
        encoded = encoder('fwd', x=x1, lengths=len1, causal=False).transpose(0, 1)

    # Convert the input and label sequences to sets
    input_set = convert_token_sequence_to_list_of_lists(input_sequence)
    label_set = convert_token_sequence_to_list_of_lists(label_sequence)
    print("input set:", input_set)
    print("label set:", label_set)

    # Initialize flags to check if any hypothesis is valid for both beam sizes
    is_valid_solution_found_beam1 = False
    is_valid_solution_found_beam10 = False

input set: [[2], [1], [0], [0, 1, 2, 3]]
label set: [[0, 1, 2, 3]]


### Decode with beam search

In [219]:
beam_size = 1
total_valid_answers_beam1 = 0
with torch.no_grad():
    _, _, beam = decoder.generate_beam(encoded, len1, beam_size=1, length_penalty=1.0, early_stopping=1, max_len=200)
    assert len(beam) == 1
hypotheses_beam1 = beam[0].hyp
assert len(hypotheses_beam1) == beam_size

# Iterate over hypotheses for beam size 1
for j, (score, sent) in enumerate(sorted(hypotheses_beam1, key=lambda x: x[0], reverse=True)):
        # Parse decoded hypothesis
    ids = sent[1:].tolist()  # decoded token IDs
    tok = [env.id2word[wid] for wid in ids]  # convert to prefix

    try:
        hyp = convert_token_sequence_to_list_of_lists(tok)
    except:
        hyp = tok
    print("beam1 answer:","%.5f  %s" % (score, hyp))

    # Validate the solution
    valid_solution_count, error_counts, coverage_percentage = validate_set_cover_solution(input_set, hyp, label_set)

    # Check if the hypothesis is a valid solution
    if valid_solution_count > 0:
        is_valid_solution_found_beam1 = True
        break  # No need to check further hypotheses

# Count the problem set as having a valid answer for beam size 1 if any hypothesis is valid
if is_valid_solution_found_beam1:
    total_valid_answers_beam1 += 1
print(f"Total number of valid answers with beam size 1: {total_valid_answers_beam1} out of {num_iterations}")

beam1 answer: -0.00002  [[0, 1, 2, 3]]
Total number of valid answers with beam size 1: 1 out of 1


In [223]:
# Decode with beam size 10
total_valid_answers_beam10 = 0
beam_size = 1000
is_valid_solution_found_beam10 = False
with torch.no_grad():
    _, _, beam = decoder.generate_beam(encoded, len1, beam_size=beam_size, length_penalty=1.0, early_stopping=1, max_len=200)
    assert len(beam) == 1
hypotheses_beam10 = beam[0].hyp
assert len(hypotheses_beam10) == beam_size

    # Iterate over hypotheses for beam size 10
for j, (score, sent) in enumerate(sorted(hypotheses_beam10, key=lambda x: x[0], reverse=True)):
        # Parse decoded hypothesis
    ids = sent[1:].tolist()  # decoded token IDs
    tok = [env.id2word[wid] for wid in ids]  # convert to prefix

    try:
        hyp = convert_token_sequence_to_list_of_lists(tok)
    except:
        hyp = tok
            
    print("beam10 answer:",j,"%.5f  %s \t %s" % (score, hyp, ids))

    # Validate the solution
    valid_solution_count, error_counts, coverage_percentage = validate_set_cover_solution(input_set, hyp, label_set)

    # Check if the hypothesis is a valid solution
    if valid_solution_count > 0:
        is_valid_solution_found_beam10 = True
    #    break  # No need to check further hypotheses

    # Count the problem set as having a valid answer for beam size 10 if any hypothesis is valid
if is_valid_solution_found_beam10:
    total_valid_answers_beam10 += 1

# Print the total number of valid answers for both beam sizes
print(f"Total number of valid answers with beam size 10: {total_valid_answers_beam10} out of {num_iterations}")

beam10 answer: 0 -0.00002  [[0, 1, 2, 3]] 	 [5, 5, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 1 -0.67432  [] 	 [5, 5, 81, 81, 7, 5, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 2 -0.73177  [[0, 1, 2, 3]] 	 [5, 81, 7, 5, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 3 -0.77823  [[0, 1, 2, 3, 3]] 	 [5, 5, 81, 7, 82, 7, 83, 7, 84, 7, 84, 6, 6]
beam10 answer: 4 -0.81957  [] 	 [5, 5, 81, 7, 5, 81, 7, 5, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 5 -0.82740  [[0, 0, 0, 1, 2, 3]] 	 [5, 5, 81, 81, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 6 -0.85051  [[0, 5], [0, 1, 2, 3]] 	 [5, 5, 81, 7, 86, 6, 7, 5, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 7 -0.87236  [] 	 [5, 5, 81, 7, 82, 7, 83, 5, 81, 7, 82, 7, 83, 7, 84, 6, 6]
beam10 answer: 8 -0.88624  [[0, 1, 2, 0, 3]] 	 [5, 5, 81, 7, 82, 7, 83, 7, 81, 7, 84, 6, 6]
beam10 answer: 9 -0.89063  [[0, 1, 3], [0, 2, 3]] 	 [5, 5, 81, 7, 82, 7, 84, 6, 7, 5, 81, 7, 83, 7, 84, 6, 6]
beam10 answer: 10 -0.89149  [] 	 [5, 5, 81, 7, 7, 5, 81, 7, 82, 

In [224]:
env.id2word

{0: '<s>',
 1: '</s>',
 2: '<pad>',
 3: '(',
 4: ')',
 5: '[',
 6: ']',
 7: ',',
 8: '<SPECIAL_8>',
 9: '<SPECIAL_9>',
 10: 'pi',
 11: 'E',
 12: 'x',
 13: 'y',
 14: 'z',
 15: 't',
 16: 'a0',
 17: 'a1',
 18: 'a2',
 19: 'a3',
 20: 'a4',
 21: 'a5',
 22: 'a6',
 23: 'a7',
 24: 'a8',
 25: 'a9',
 26: 'abs',
 27: 'acos',
 28: 'acosh',
 29: 'acot',
 30: 'acoth',
 31: 'acsc',
 32: 'acsch',
 33: 'add',
 34: 'asec',
 35: 'asech',
 36: 'asin',
 37: 'asinh',
 38: 'atan',
 39: 'atanh',
 40: 'cos',
 41: 'cosh',
 42: 'cot',
 43: 'coth',
 44: 'csc',
 45: 'csch',
 46: 'derivative',
 47: 'div',
 48: 'exp',
 49: 'f',
 50: 'g',
 51: 'h',
 52: 'inv',
 53: 'ln',
 54: 'mul',
 55: 'pow',
 56: 'pow2',
 57: 'pow3',
 58: 'pow4',
 59: 'pow5',
 60: 'rac',
 61: 'sec',
 62: 'sech',
 63: 'sign',
 64: 'sin',
 65: 'sinh',
 66: 'sqrt',
 67: 'sub',
 68: 'tan',
 69: 'tanh',
 70: 'I',
 71: 'INT+',
 72: 'INT-',
 73: 'INT',
 74: 'FLOAT',
 75: '-',
 76: '.',
 77: '10^',
 78: 'Y',
 79: "Y'",
 80: "Y''",
 81: '0',
 82: '1',
 83: 

In [197]:
#Saving to the files

Notes for me:
- The hypotheses variable is a list of tuples, where each tuple contains a score and a corresponding sentence (sequence of token IDs).
- The sorted function sorts the list of hypotheses based on the score (x[0]) in descending order (reverse=True).
- The for loop iterates over each sorted hypothesis.
